<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/02_alert_rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

df = pd.read_parquet('/content/drive/MyDrive/aml-project/sample.parquet')

print(f"{len(df):,} rows | {df['Is_laundering'].sum():,} criminal "
      f"({df['Is_laundering'].mean()*100:.3f}%)")
print(f"{df['Date'].min()} to {df['Date'].max()}")

Mounted at /content/drive
1,509,825 rows | 9,873 criminal (0.654%)
2022-10-07 to 2023-08-23


In [2]:
total = len(df)
crime = df['Is_laundering'].sum()

print(f"Population:      {total:,}")
print(f"Criminal:        {crime:,}")
print(f"Base rate:       {crime/total*100:.3f}%")
print(f"\nIf we alerted on EVERYTHING:")
print(f"  alerts:        {total:,}")
print(f"  recall:        100.0%")
print(f"  precision:     {crime/total*100:.3f}%")

Population:      1,509,825
Criminal:        9,873
Base rate:       0.654%

If we alerted on EVERYTHING:
  alerts:        1,509,825
  recall:        100.0%
  precision:     0.654%


In [7]:
def score_rule(mask, name, df=df):
    n_alerts = mask.sum()
    if n_alerts == 0:
        print(f"{name}: fires on nothing")
        return None
    caught = df.loc[mask, 'Is_laundering'].sum()
    precision = caught / n_alerts * 100
    recall = caught / df['Is_laundering'].sum() * 100
    lift = precision / (df['Is_laundering'].mean() * 100)

    print(f"--- {name} ---")
    print(f"  alerts:     {n_alerts:>9,}  ({n_alerts/len(df)*100:.2f}% of all txns)")
    print(f"  caught:     {caught:>9,}")
    print(f"  precision:  {precision:>9.3f}%")
    print(f"  recall:     {recall:>9.2f}%")
    print(f"  lift:       {lift:>9.1f}x baseline")
    return {'rule': name, 'alerts': n_alerts, 'caught': caught,
            'precision': precision, 'recall': recall, 'lift': lift}

results = []

In [8]:
r1 = (df['Payment_type'] == 'Cash Withdrawal') & (df['Amount'] < 400)
results.append(score_rule(r1, "R1: cash withdrawal under 400"))

--- R1: cash withdrawal under 400 ---
  alerts:        48,771  (3.23% of all txns)
  caught:         1,334
  precision:      2.735%
  recall:         13.51%
  lift:             4.2x baseline


In [9]:
r2 = df['Amount'] > 50000
results.append(score_rule(r2, "R2: amount over 50k"))

--- R2: amount over 50k ---
  alerts:        12,824  (0.85% of all txns)
  caught:           402
  precision:      3.135%
  recall:          4.07%
  lift:             4.8x baseline


In [10]:
r3 = (df['Sender_bank_location'] != df['Receiver_bank_location'])
results.append(score_rule(r3, "R3: cross-border (any)"))

--- R3: cross-border (any) ---
  alerts:       150,057  (9.94% of all txns)
  caught:         3,045
  precision:      2.029%
  recall:         30.84%
  lift:             3.1x baseline


In [11]:
r4 = (df['Amount'] >= 3000) & (df['Amount'] <= 7000) & \
     (df['Payment_type'].isin(['Cash Deposit', 'Cash Withdrawal']))
results.append(score_rule(r4, "R4: mid-range cash 3k-7k"))

--- R4: mid-range cash 3k-7k ---
  alerts:        10,120  (0.67% of all txns)
  caught:           498
  precision:      4.921%
  recall:          5.04%
  lift:             7.5x baseline


In [12]:
df['alert_r1'] = r1.astype(int)
df['alert_r2'] = r2.astype(int)
df['alert_r3'] = r3.astype(int)
df['alert_r4'] = r4.astype(int)
df['n_rules'] = df[['alert_r1','alert_r2','alert_r3','alert_r4']].sum(axis=1)
df['alerted'] = (df['n_rules'] > 0).astype(int)

results.append(score_rule(df['alerted'].astype(bool), "COMBINED (any rule)"))

--- COMBINED (any rule) ---
  alerts:       219,963  (14.57% of all txns)
  caught:         4,771
  precision:      2.169%
  recall:         48.32%
  lift:             3.3x baseline


In [13]:
summary = pd.DataFrame(results).set_index('rule')
print(summary.round(3))

print(f"\nOverlap — transactions by number of rules fired:")
print(df['n_rules'].value_counts().sort_index())

                               alerts  caught  precision  recall   lift
rule                                                                   
R1: cash withdrawal under 400   48771    1334      2.735  13.512  4.183
R2: amount over 50k             12824     402      3.135   4.072  4.794
R3: cross-border (any)         150057    3045      2.029  30.842  3.103
R4: mid-range cash 3k-7k        10120     498      4.921   5.044  7.525
COMBINED (any rule)            219963    4771      2.169  48.324  3.317

Overlap — transactions by number of rules fired:
n_rules
0    1289862
1     218154
2       1809
Name: count, dtype: int64


In [14]:
queue = df[df['alerted'] == 1].copy().reset_index(drop=True)

print(f"Queue size:     {len(queue):,}")
print(f"Criminal:       {queue['Is_laundering'].sum():,}")
print(f"Precision:      {queue['Is_laundering'].mean()*100:.3f}%")
print(f"\nMissed entirely: {9873 - queue['Is_laundering'].sum():,} criminal txns")

print("\nWhich patterns do the rules catch?")
caught = queue[queue['Is_laundering']==1]['Laundering_type'].value_counts()
allc = df[df['Is_laundering']==1]['Laundering_type'].value_counts()
cov = pd.DataFrame({'caught': caught, 'total': allc})
cov['coverage_%'] = (cov['caught'].fillna(0)/cov['total']*100).round(1)
print(cov.sort_values('coverage_%', ascending=False))

Queue size:     219,963
Criminal:       4,771
Precision:      2.169%

Missed entirely: 5,102 criminal txns

Which patterns do the rules catch?
                      caught  total  coverage_%
Laundering_type                                
Behavioural_Change_2     345    345       100.0
Over-Invoicing            54     54       100.0
Cash_Withdrawal         1334   1334       100.0
Single_large             235    250        94.0
Behavioural_Change_1     258    394        65.5
Deposit-Send             493    945        52.2
Cycle                    189    382        49.5
Structuring              723   1870        38.7
Scatter-Gather           130    338        38.5
Fan_Out                   90    237        38.0
Smurfing                 346    932        37.1
Layered_Fan_In           167    656        25.5
Layered_Fan_Out          134    529        25.3
Bipartite                 94    383        24.5
Stacked Bipartite         77    506        15.2
Gather-Scatter            52    354      

In [15]:
queue.to_parquet('/content/drive/MyDrive/aml-project/alert_queue.parquet', index=False)
summary.to_csv('/content/drive/MyDrive/aml-project/rule_performance.csv')
print("Saved queue and rule performance.")

Saved queue and rule performance.
